# 03 · Data preparation (Stage 2)
Cleaning, targets, temporal split, M1 features and M2 time series.

In [1]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Clone or update the project repository
import os
from google.colab import userdata
token = userdata.get('GH_TOKEN')
REPO = '/content/tevzi'
if not os.path.exists(REPO):
    !git clone -q https://{token}@github.com/emirsalihoglu/tevzi.git {REPO}
else:
    !git -C {REPO} pull -q
!git -C {REPO} log --oneline -1

a9abb69 (HEAD -> main, origin/main, origin/HEAD) feat: add temporal split with purging and burst flag


In [3]:
# Make the package importable and prepare Drive folders
import sys
from pathlib import Path
sys.path.insert(0, f'{REPO}/src')
DRIVE = Path('/content/drive/MyDrive/Tevzi')
for sub in ['data/raw', 'data/interim', 'data/processed',
            'results/models', 'results/figures', 'results/tables']:
    (DRIVE / sub).mkdir(parents=True, exist_ok=True)
import tevzi
print('tevzi package:', tevzi.__file__)
print('Drive folder:', DRIVE)

tevzi package: /content/tevzi/src/tevzi/__init__.py
Drive folder: /content/drive/MyDrive/Tevzi


## 2.1 Remaining exploration checks
Can we recover run intervals (first start, last end) of Failed and Running jobs from the instance table? Required for using them as M2 background load (decision C).

In [4]:
# Show column names of the tables used below (schema only, no data loaded)
from tevzi.data.loader import table_columns
INTERIM = DRIVE / 'data/interim'
for name in ['job', 'instance']:
    print(name, '->', table_columns(name, INTERIM))

job -> ['job_name', 'inst_id', 'user', 'status', 'start_time', 'end_time']
instance -> ['job_name', 'task_name', 'inst_name', 'worker_name', 'inst_id', 'status', 'start_time', 'end_time', 'machine']


In [5]:
# Per-job run interval from instance attempts, summarised by job status
import gc
import pandas as pd
import psutil
from tevzi.data.loader import load_table

job = load_table('job', INTERIM, columns=['job_name', 'status', 'start_time', 'end_time'])
inst = load_table('instance', INTERIM, columns=['job_name', 'start_time', 'end_time'])
print('instance end_time == 0:', int((inst['end_time'] == 0).sum()))

agg = inst.groupby('job_name', sort=False).agg(
    inst_start_min=('start_time', 'min'),
    inst_end_max=('end_time', 'max'),
)
del inst; gc.collect()

df = job.merge(agg, left_on='job_name', right_index=True, how='left')
del agg; gc.collect()

def pct(mask):
    return f'{100 * mask.mean():.1f}%'

rows = []
for status, g in df.groupby('status'):
    rows.append({
        'status': status,
        'jobs': len(g),
        'job_end': pct(g['end_time'].notna()),
        'inst_start': pct(g['inst_start_min'].notna()),
        'inst_end': pct(g['inst_end_max'].notna()),
        'any_end': pct(g['end_time'].notna() | g['inst_end_max'].notna()),
    })
print(pd.DataFrame(rows).to_string(index=False))

# Failed jobs: run length from instances, and agreement with job end where both exist
f = df[df['status'] == 'Failed']
run = (f['inst_end_max'] - f['inst_start_min']).dropna()
print('\nFailed run length (s) quantiles:')
print(run.quantile([0.01, 0.25, 0.5, 0.75, 0.99]).round(0).to_string())
both = f.dropna(subset=['end_time', 'inst_end_max'])
print('Failed jobs with both ends:', len(both),
      '| median |job_end - inst_end|:', (both['end_time'] - both['inst_end_max']).abs().median())

print(f'\nRAM used: {psutil.virtual_memory().used / 1e9:.2f} GB')

instance end_time == 0: 0
    status   jobs job_end inst_start inst_end any_end
    Failed 256555   10.0%     100.0%    11.8%   15.7%
   Running  62928    1.7%      25.9%    12.0%   12.4%
Terminated 732355  100.0%     100.0%   100.0%  100.0%
   Waiting   3663    1.2%      14.6%     4.9%    5.1%

Failed run length (s) quantiles:
0.01       -41.0
0.25        23.0
0.50       111.0
0.75       580.0
0.99    137298.0
Failed jobs with both ends: 15701 | median |job_end - inst_end|: 0.0

RAM used: 1.75 GB


In [6]:
# Can machine_metric recover end times of Failed / Running jobs?
mm_cols = table_columns('machine_metric', INTERIM)
print('machine_metric ->', mm_cols)
assert {'worker_name', 'start_time', 'end_time'} <= set(mm_cols), 'column names differ: send this output'

mm = load_table('machine_metric', INTERIM, columns=['worker_name', 'start_time', 'end_time'])
print('rows:', len(mm), '| unique workers:', mm['worker_name'].nunique())
print(f"missing start / end: {100 * mm['start_time'].isna().mean():.1f}% / "
      f"{100 * mm['end_time'].isna().mean():.1f}%")
mm_w = mm.groupby('worker_name', sort=False).agg(n_mm=('start_time', 'size'),
                                                 mm_end=('end_time', 'max'))
del mm; gc.collect()

inst = load_table('instance', INTERIM, columns=['job_name', 'worker_name'])
w = inst.merge(mm_w, left_on='worker_name', right_index=True, how='left')
del inst, mm_w; gc.collect()
w['in_mm'] = w['n_mm'].notna()
wj = w.groupby('job_name', sort=False).agg(in_mm=('in_mm', 'any'), mm_end=('mm_end', 'max'))
del w; gc.collect()

df2 = df.merge(wj, left_on='job_name', right_index=True, how='left')
del wj; gc.collect()

rows = []
for status, g in df2.groupby('status'):
    known_before = g['end_time'].notna() | g['inst_end_max'].notna()
    rows.append({
        'status': status,
        'jobs': len(g),
        'in_machine_metric': pct(g['in_mm'].fillna(False)),
        'mm_end': pct(g['mm_end'].notna()),
        'any_end_before': pct(known_before),
        'any_end_with_mm': pct(known_before | g['mm_end'].notna()),
    })
print(pd.DataFrame(rows).to_string(index=False))

both = df2[(df2['status'] == 'Failed')].dropna(subset=['inst_end_max', 'mm_end'])
print('\nFailed with instance end and mm end:', len(both),
      '| median |inst_end - mm_end|:', (both['inst_end_max'] - both['mm_end']).abs().median())

machine_metric -> ['worker_name', 'machine', 'start_time', 'end_time', 'machine_cpu_iowait', 'machine_cpu_kernel', 'machine_cpu_usr', 'machine_gpu', 'machine_load_1', 'machine_net_receive', 'machine_num_worker', 'machine_cpu']
rows: 2009423 | unique workers: 2009421
missing start / end: 0.0% / 0.0%
    status   jobs in_machine_metric mm_end any_end_before any_end_with_mm
    Failed 256555              0.9%   0.9%          15.7%           15.7%
   Running  62928              5.2%   5.2%          12.4%           12.4%
Terminated 732355             78.5%  78.5%         100.0%          100.0%
   Waiting   3663              0.0%   0.0%           5.1%            5.1%

Failed with instance end and mm end: 2202 | median |inst_end - mm_end|: 1.0


/tmp/ipykernel_19229/4148520953.py:30: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  'in_machine_metric': pct(g['in_mm'].fillna(False)),


In [7]:
# When were the 'Running' jobs submitted? (days since first submission, t0 = 494319 s)
T0 = 494319
r_days = (df2.loc[df2['status'] == 'Running', 'start_time'] - T0) / 86400
print(r_days.describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1).to_string())
print(f"\nsubmitted in the last 2 days of the trace: {100 * (r_days > r_days.max() - 2).mean():.1f}%")
print(f'RAM used: {psutil.virtual_memory().used / 1e9:.2f} GB')

count    62928.0
mean        37.2
std         16.9
min          0.3
10%         13.7
25%         22.8
50%         36.7
75%         50.8
90%         60.7
max         68.9

submitted in the last 2 days of the trace: 1.4%
RAM used: 2.54 GB


## 2.2 CPU-only tasks and machine_metric content
Do CPU-only tasks have sensor records, and do they run on GPU machines? Is machine_metric usable as a usage-based occupancy signal?

In [8]:
# CPU-only tasks: sensor coverage and machine types they run on
import gc
import pandas as pd
import psutil
from tevzi.data.loader import load_table, table_columns
INTERIM = DRIVE / 'data/interim'

cols = {n: table_columns(n, INTERIM) for n in ['task', 'sensor', 'machine_spec']}
for n, c in cols.items():
    print(n, '->', c)
assert {'job_name', 'task_name', 'gpu_type'} <= set(cols['task']), 'task columns differ'
assert {'job_name', 'task_name'} <= set(cols['sensor']), 'sensor columns differ'
assert {'machine', 'gpu_type'} <= set(cols['machine_spec']), 'machine_spec columns differ'
for name in ['df', 'df2']:              # free objects from earlier cells, if present
    globals().pop(name, None)
gc.collect()

task = load_table('task', INTERIM, columns=['job_name', 'task_name', 'gpu_type'])
print('\ntask gpu_type values:')
print(task['gpu_type'].value_counts(dropna=False).to_string())
task['cpu_only'] = task['gpu_type'].isna() | (task['gpu_type'].astype(str) == 'None')

# Sensor coverage per task
sens = load_table('sensor', INTERIM, columns=['job_name', 'task_name']).drop_duplicates()
t = task.merge(sens, on=['job_name', 'task_name'], how='left', indicator=True)
t['has_sensor'] = t['_merge'].eq('both')
del sens; gc.collect()
job = load_table('job', INTERIM, columns=['job_name', 'status'])
t = t.merge(job, on='job_name', how='left')
del job; gc.collect()
summary = (t.assign(terminated=t['status'].eq('Terminated'))
             .groupby(['cpu_only', 'terminated'])['has_sensor']
             .agg(tasks='size', with_sensor='mean'))
summary['with_sensor'] = (100 * summary['with_sensor']).round(1)
print('\nSensor coverage (%):')
print(summary.to_string())
del t; gc.collect()

# Which machine types do instances of CPU-only vs GPU tasks run on?
spec = load_table('machine_spec', INTERIM, columns=['machine', 'gpu_type'])
inst = load_table('instance', INTERIM, columns=['job_name', 'task_name', 'machine'])
inst = inst.merge(task[['job_name', 'task_name', 'cpu_only']], on=['job_name', 'task_name'], how='left')
inst = inst.merge(spec.rename(columns={'gpu_type': 'machine_type'}), on='machine', how='left')
print('\nInstance share by machine type (rows = task kind):')
print(pd.crosstab(inst['cpu_only'], inst['machine_type'].fillna('unknown'), normalize='index').round(3).to_string())
del inst, spec; gc.collect()
print(f'\nRAM used: {psutil.virtual_memory().used / 1e9:.2f} GB')

task -> ['job_name', 'task_name', 'inst_num', 'status', 'start_time', 'end_time', 'plan_cpu', 'plan_mem', 'plan_gpu', 'gpu_type']
sensor -> ['job_name', 'task_name', 'worker_name', 'inst_id', 'machine', 'gpu_name', 'cpu_usage', 'gpu_wrk_util', 'avg_mem', 'max_mem', 'avg_gpu_wrk_mem', 'max_gpu_wrk_mem', 'read', 'write', 'read_count', 'write_count']
machine_spec -> ['machine', 'gpu_type', 'cap_cpu', 'cap_mem', 'cap_gpu']

task gpu_type values:
gpu_type
MISC       696281
T4         226923
None       217738
P100        72732
V100        28797
V100M32     18579

Sensor coverage (%):
                      tasks  with_sensor
cpu_only terminated                     
False    False       310994         84.5
         True        732318         80.2
True     False        66820          0.0
         True        150918          0.0

Instance share by machine type (rows = task kind):
machine_type   CPU   MISC   P100     T4   V100  V100M32  unknown
cpu_only                                            

In [9]:
# machine_metric content: value ranges, row durations, how regularly machines are observed
T0 = 494319
mm = load_table('machine_metric', INTERIM,
                columns=['machine', 'start_time', 'end_time',
                         'machine_gpu', 'machine_cpu', 'machine_num_worker'])
print(mm[['machine_gpu', 'machine_cpu', 'machine_num_worker']]
      .describe(percentiles=[0.5, 0.9, 0.99]).round(2).to_string())
dur = mm['end_time'] - mm['start_time']
print('\nrow duration (s) quantiles:', dur.quantile([0.5, 0.9, 0.99]).round(0).to_dict())
print('distinct machines:', mm['machine'].nunique())
day = ((mm['start_time'] - T0) // 86400).astype(int)
per_day = mm.groupby(day)['machine'].nunique()
print('machines observed per day: median', per_day.median(), '| min', per_day.min(), '| max', per_day.max())
del mm, dur, day; gc.collect()
print(f'RAM used: {psutil.virtual_memory().used / 1e9:.2f} GB')

       machine_gpu  machine_cpu  machine_num_worker
count   1767929.00    883528.00          1714165.00
mean        135.10        27.61               11.79
std          96.57        18.66                6.76
min           0.00         0.72                0.00
50%         124.37        22.79               12.88
90%         259.52        54.55               20.24
99%         428.43        83.37               24.20
max         787.41        99.27               29.75

row duration (s) quantiles: {0.5: 1241.0, 0.9: 16437.0, 0.99: 82503.0}
distinct machines: 1720
machines observed per day: median 1448.0 | min 1 | max 1561
RAM used: 2.47 GB


## 3. Cleaning: exclusion rules and analysis sets
Rules R1–R7 are stored as flags (nothing is deleted). Sets: m2_arrival, m2_occupancy, model.

In [10]:
# Run the cleaning and show the exclusion flow table
import time
import gc
import pandas as pd
import psutil
from tevzi.data.cleaning import clean

INTERIM = DRIVE / 'data/interim'
t0 = time.time()
jobs, flow = clean(INTERIM)
gc.collect()
print(f'done in {time.time() - t0:.0f} s | jobs: {len(jobs):,} | '
      f'RAM used: {psutil.virtual_memory().used / 1e9:.2f} GB\n')
print(flow.to_string(index=False))

done in 10 s | jobs: 1,055,501 | RAM used: 2.62 GB

                    step                                                 description  removed  remaining  flagged_any_order
                All jobs                                                                    0    1055501                  0
       R1_not_terminated                                Job status is not Terminated   323146     732355             323146
             R2_cpu_only                                      No task requests a GPU    17452     714903              31514
      R3_missing_request    Missing CPU/memory request, or GPU type without plan_gpu        0     714903              16911
     R4_zero_cpu_request                                       A task requests 0 CPU        0     714903                 10
R5_request_over_capacity A request exceeds the largest machine of the requested type       83     714820                169
    R6_time_inconsistent          Run starts before submission, or run duration 

In [11]:
# Set sizes, sanity checks and save the flow table
print('Set sizes:')
print(jobs[['m2_arrival', 'm2_occupancy', 'model']].sum().to_string())

print('\nm2_occupancy by status:')
print(pd.crosstab(jobs['status'], jobs['m2_occupancy']).to_string())

print('\nm2_occupancy: CPU-only vs GPU jobs:')
occ = jobs[jobs['m2_occupancy']]
print(occ['R2_cpu_only'].value_counts().rename({True: 'cpu_only', False: 'gpu'}).to_string())

run = (jobs.loc[jobs['model'], 'end_time'] - jobs.loc[jobs['model'], 'first_start'])
print('\nmodel set run duration (s) quantiles:')
print(run.quantile([0.01, 0.25, 0.5, 0.75, 0.99]).round(0).to_string())

out = DRIVE / 'results/tables/exclusion_flow.csv'
flow.to_csv(out, index=False)
print('\nsaved:', out)

Set sizes:
m2_arrival      1051838
m2_occupancy     732257
model            714273

m2_occupancy by status:
m2_occupancy   False   True 
status                      
Failed        256555       0
Running        62928       0
Terminated        98  732257
Waiting         3663       0

m2_occupancy: CPU-only vs GPU jobs:
R2_cpu_only
gpu         714813
cpu_only     17444

model set run duration (s) quantiles:
0.01       12.0
0.25      103.0
0.50      534.0
0.75     3550.0
0.99    65199.0

saved: /content/drive/MyDrive/Tevzi/results/tables/exclusion_flow.csv


In [12]:
# Check 1: duration definitions (with vs without queueing) on Terminated jobs
for name in ['occ', 'run']:
    globals().pop(name, None)
gc.collect()

term = jobs[jobs['status'].eq('Terminated')]
d = pd.DataFrame({
    'end_minus_submit': term['end_time'] - term['submit_time'],
    'end_minus_first_start': term['end_time'] - term['first_start'],
})
print('Terminated jobs, duration quantiles (s):')
print(d.quantile([0.01, 0.25, 0.5, 0.75, 0.99]).round(0).to_string())
del term, d; gc.collect()

# Check 2: which tasks are CPU-only, and do they sit inside GPU jobs?
from tevzi.data.loader import load_table
task = load_table('task', INTERIM, columns=['job_name', 'task_name', 'gpu_type'])
cpu_t = task[task['gpu_type'].astype(str).eq('None')]
cpu_t = cpu_t.merge(jobs[['job_name', 'R2_cpu_only']], on='job_name', how='left')
kind = cpu_t['R2_cpu_only'].map({True: 'cpu_only_job', False: 'gpu_job'})
print('\nCPU-only tasks by task name and job kind (top 10):')
print(pd.crosstab(cpu_t['task_name'], kind).sort_values('gpu_job', ascending=False).head(10).to_string())
del task, cpu_t, kind; gc.collect()
print(f'\nRAM used: {psutil.virtual_memory().used / 1e9:.2f} GB')

Terminated jobs, duration quantiles (s):
      end_minus_submit  end_minus_first_start
0.01              17.0                   11.0
0.25             144.0                  104.0
0.50             663.0                  551.0
0.75            3871.0                 3571.0
0.99           68884.0                66153.0

CPU-only tasks by task name and job kind (top 10):
R2_cpu_only     cpu_only_job  gpu_job
task_name                            
ps                      4719   172335
evaluator                127     3216
OpenmpiTracker           114      631
worker                  5541      581
chief                     11      344
aligraph                   0        6
M2                         1        0
JupyterTask             1618        0
DecoderWorker           4136        0
BladeMain                  1        0

RAM used: 2.54 GB


## 4. Job table and targets
Main task = task with the largest total GPU request (plan_gpu × inst_num); ties → higher plan_gpu, then task name.

In [13]:
# How often do the two main-task definitions disagree? (model set, GPU-requesting tasks only)
from tevzi.data.loader import load_table
model_jobs = jobs.loc[jobs['model'], 'job_name']
task = load_table('task', INTERIM, columns=['job_name', 'task_name', 'inst_num', 'plan_gpu'])
task = task[task['job_name'].isin(model_jobs) & task['plan_gpu'].gt(0)].copy()
task['gpu_total'] = task['plan_gpu'] * task['inst_num']

n_gpu_tasks = task.groupby('job_name').size()
print(f'model jobs with >1 GPU task: {(n_gpu_tasks > 1).sum():,} ({100 * (n_gpu_tasks > 1).mean():.2f}%)')

def pick(df, keys):
    order = df.sort_values(['job_name'] + keys + ['task_name'],
                           ascending=[True] + [False] * len(keys) + [True])
    return order.drop_duplicates('job_name').set_index('job_name')['task_name']

a = pick(task, ['plan_gpu', 'gpu_total'])   # A: largest per-instance request
b = pick(task, ['gpu_total', 'plan_gpu'])   # B: largest total request
diff = a.ne(b.reindex(a.index))
print(f'jobs where A and B pick a different main task: {diff.sum():,}')

multi = task[task['job_name'].isin(n_gpu_tasks[n_gpu_tasks > 1].index)]
print('\nGPU task combinations in multi-GPU-task jobs (top 8):')
print(multi.groupby('job_name')['task_name'].agg(lambda s: '+'.join(sorted(s))).value_counts().head(8).to_string())
print('\nB picks in those jobs:')
print(b.reindex(multi['job_name'].unique()).value_counts().head(8).to_string())
del task, multi; gc.collect()

model jobs with >1 GPU task: 17,415 (2.44%)
jobs where A and B pick a different main task: 83

GPU task combinations in multi-GPU-task jobs (top 8):
task_name
evaluator+worker    12479
ps+worker            4929
chief+worker            7

B picks in those jobs:
task_name
worker       8479
evaluator    5110
ps           3823
chief           3


0

In [14]:
# Are evaluator/ps picks caused by ties? Compare with a role-aware tie-break
ROLE_RANK = {'worker': 0, 'chief': 1, 'evaluator': 2, 'ps': 3}   # lower = preferred on ties

task = load_table('task', INTERIM, columns=['job_name', 'task_name', 'inst_num', 'plan_gpu'])
task = task[task['job_name'].isin(model_jobs) & task['plan_gpu'].gt(0)].copy()
task['gpu_total'] = task['plan_gpu'] * task['inst_num']
task['role_rank'] = task['task_name'].map(ROLE_RANK).fillna(0)   # other names behave like worker

top = task.groupby('job_name')['gpu_total'].transform('max')
cand = task[task['gpu_total'].eq(top)]
n_cand = cand.groupby('job_name').size()
tied = n_cand[n_cand > 1].index
print(f'jobs with a tie on total GPU: {len(tied):,}')
print(cand[cand['job_name'].isin(tied)].groupby('job_name')['task_name']
      .agg(lambda s: '+'.join(sorted(s))).value_counts().head(5).to_string())

order = task.sort_values(['job_name', 'gpu_total', 'plan_gpu', 'role_rank', 'task_name'],
                         ascending=[True, False, False, True, True])
c = order.drop_duplicates('job_name').set_index('job_name')['task_name']
multi_ids = n_gpu_tasks[n_gpu_tasks > 1].index
print('\nRole-aware picks in multi-GPU-task jobs:')
print(c.reindex(multi_ids).value_counts().to_string())
print(f'\njobs where role-aware rule differs from B: {c.ne(b.reindex(c.index)).sum():,}')
del task, cand, order; gc.collect();

jobs with a tie on total GPU: 8,360
task_name
evaluator+worker    5110
ps+worker           3247
chief+worker           3

Role-aware picks in multi-GPU-task jobs:
task_name
worker       16837
ps             576
evaluator        2

jobs where role-aware rule differs from B: 8,358


In [15]:
# Pull the new module, free leftovers from cells 18-19, build targets for the model set
!git -C {REPO} pull -q
!git -C {REPO} log --oneline -1
for name in ['a', 'b', 'c', 'diff', 'n_gpu_tasks', 'multi_ids', 'model_jobs',
             'top', 'n_cand', 'tied', 'ROLE_RANK']:
    globals().pop(name, None)
gc.collect()

from tevzi.data.targets import build_targets
t0 = time.time()
targets = build_targets(jobs, INTERIM)
gc.collect()
print(f'done in {time.time() - t0:.0f} s | rows: {len(targets):,} | '
      f'missing main task: {targets["main_task"].isna().sum()} | '
      f'RAM used: {psutil.virtual_memory().used / 1e9:.2f} GB')
print('\nmain task names (top 8):')
print(targets['main_task'].value_counts().head(8).to_string())
print(f"\nhas_usage: {targets['has_usage'].sum():,} jobs ({100 * targets['has_usage'].mean():.1f}%)")

a9abb69 (HEAD -> main, origin/main, origin/HEAD) feat: add temporal split with purging and burst flag
done in 15 s | rows: 714,273 | missing main task: 0 | RAM used: 2.80 GB

main task names (top 8):
main_task
tensorflow           406875
worker               210865
PyTorchWorker         69714
xComputeWorker        23411
TVMTuneMain            1158
ps                      774
OssToVolumeWorker       639
OpenmpiWorker           521

has_usage: 575,254 jobs (80.5%)


In [16]:
# Target distributions and sanity checks
q = [0.01, 0.25, 0.5, 0.75, 0.9, 0.99]
u = targets[targets['has_usage']]
summary = pd.DataFrame({
    'run_time_s': targets['run_time'].quantile(q),
    'wait_time_s': targets['wait_time'].quantile(q),
    'gpu_ratio': u['gpu_ratio'].quantile(q),
    'mem_ratio': u['mem_ratio'].quantile(q),
    'usage_coverage': u['usage_coverage'].quantile(q),
})
print(summary.round(3).to_string())

print(f"\ngpu_ratio > 1 (uses more than requested): {100 * (u['gpu_ratio'] > 1).mean():.1f}%")
print(f"mem_ratio > 1 (uses more than requested): {100 * (u['mem_ratio'] > 1).mean():.1f}%")
print(f"gpu_ratio == 0 (GPU never used):          {100 * (u['gpu_ratio'] == 0).mean():.1f}%")
print(f"usage_coverage == 1 (all attempts seen):  {100 * (u['usage_coverage'] == 1).mean():.1f}%")
del u; gc.collect();

      run_time_s  wait_time_s  gpu_ratio  mem_ratio  usage_coverage
0.01        12.0          2.0      0.000      0.008             0.2
0.25       103.0          3.0      0.000      0.061             1.0
0.50       534.0          7.0      0.050      0.169             1.0
0.75      3550.0         13.0      0.332      0.529             1.0
0.90     11901.8        110.0      0.728      1.027             1.0
0.99     65198.8       5699.0      1.397      3.082             1.0

gpu_ratio > 1 (uses more than requested): 3.5%
mem_ratio > 1 (uses more than requested): 10.6%
gpu_ratio == 0 (GPU never used):          34.3%
usage_coverage == 1 (all attempts seen):  78.4%


## 5. Temporal split and purging
Periods by submission day: warm-up <7, train 7–45, val 45–52, test ≥52. Train/val jobs that end after their period are purged. Burst = days 58.8–60.5.

In [17]:
# Pull the new module, assign periods to model-set jobs, summarise kept vs purged
!git -C {REPO} pull -q
!git -C {REPO} log --oneline -1
from tevzi.data.split import assign_periods, split_summary

periods = assign_periods(jobs.loc[jobs['model'], ['job_name', 'submit_time', 'end_time']])
cols = ['submit_day', 'period', 'purged', 'in_burst']
targets = targets.drop(columns=[c for c in cols if c in targets]).merge(periods, on='job_name', how='left')
del periods; gc.collect()

summary = split_summary(targets[['job_name'] + cols], targets['run_time'])
print(summary.round(0).to_string(index=False))
out = DRIVE / 'results/tables/split_summary.csv'
summary.to_csv(out, index=False)
print('\nsaved:', out)

a9abb69 (HEAD -> main, origin/main, origin/HEAD) feat: add temporal split with purging and burst flag
period  group   jobs  run_time_median  run_time_p90  run_time_p99  in_burst
warmup   kept   1208           3508.0       75186.0      220201.0         0
 train   kept 403687            515.0       12433.0       65081.0         0
 train purged    531          42194.0      204853.0      530297.0         0
   val   kept  88611            601.0       11222.0       55184.0         0
   val purged    801          35802.0      146534.0      366509.0         0
  test   kept 219435            526.0       10646.0       61845.0     19543

saved: /content/drive/MyDrive/Tevzi/results/tables/split_summary.csv


In [18]:
# Usable jobs per period, and how purging affects long jobs (> 1 day)
use = targets[~targets['purged']]
tab = use.groupby('period').agg(jobs=('job_name', 'size'),
                                with_usage=('has_usage', 'sum'),
                                in_burst=('in_burst', 'sum'))
tab['share_%'] = (100 * tab['jobs'] / tab['jobs'].sum()).round(1)
print(tab.reindex(['warmup', 'train', 'val', 'test']).to_string())

long = targets['run_time'] > 86_400
print('\nJobs running > 1 day, by period and purge flag:')
print(pd.crosstab(targets.loc[long, 'period'], targets.loc[long, 'purged']).to_string())
del use, long; gc.collect();

          jobs  with_usage  in_burst  share_%
period                                       
warmup    1208           0         0      0.2
train   403687      272339         0     56.6
val      88611       87492         0     12.4
test    219435      214099     19543     30.8

Jobs running > 1 day, by period and purge flag:
purged  False  True 
period              
test     1334      0
train    2398    141
val       419    209
warmup     95      0


In [19]:
# Sensor coverage (has_usage) by submission day: where does it start?
import numpy as np
cov = targets.groupby(np.floor(targets['submit_day']).astype(int))['has_usage'].agg(['mean', 'size'])
cov['mean'] = (100 * cov['mean']).round(0).astype(int)
for start in range(0, len(cov), 10):
    chunk = cov.iloc[start:start + 10]
    print('  '.join(f"d{d}:{m}%" for d, m in zip(chunk.index, chunk['mean'])))
first_full = cov.index[cov['mean'] >= 90]
print('\nfirst day with >= 90% coverage:', first_full.min() if len(first_full) else None)

d0:0%  d1:0%  d2:0%  d4:0%  d5:0%  d6:0%  d7:0%  d8:0%  d9:0%  d10:0%
d11:0%  d12:0%  d13:0%  d14:0%  d15:0%  d16:0%  d17:1%  d18:15%  d19:98%  d20:98%
d21:98%  d22:97%  d23:98%  d24:98%  d25:97%  d26:97%  d27:98%  d28:97%  d29:97%  d30:95%
d31:96%  d32:97%  d33:96%  d34:94%  d35:94%  d36:95%  d37:95%  d38:96%  d39:96%  d40:96%
d41:95%  d42:96%  d43:98%  d44:97%  d45:98%  d46:99%  d47:99%  d48:99%  d49:99%  d50:99%
d51:99%  d52:99%  d53:99%  d54:99%  d55:98%  d56:99%  d57:99%  d58:99%  d59:96%  d60:96%
d61:95%  d62:96%  d63:96%  d64:97%  d65:97%  d66:97%  d67:99%  d68:99%

first day with >= 90% coverage: 19


In [20]:
# Pull the fix, reload the split module, re-assign periods with train starting on day 19
!git -C {REPO} pull -q
!git -C {REPO} log --oneline -1
import importlib
import tevzi.data.split as split_mod
importlib.reload(split_mod)
print('PERIODS:', split_mod.PERIODS)

periods = split_mod.assign_periods(jobs.loc[jobs['model'], ['job_name', 'submit_time', 'end_time']])
cols = ['submit_day', 'period', 'purged', 'in_burst']
targets = targets.drop(columns=[c for c in cols if c in targets]).merge(periods, on='job_name', how='left')
del periods; gc.collect()

summary = split_mod.split_summary(targets[['job_name'] + cols], targets['run_time'])
print('\n' + summary.round(0).to_string(index=False))
summary.to_csv(DRIVE / 'results/tables/split_summary.csv', index=False)

use = targets[~targets['purged']]
tab = use.groupby('period').agg(jobs=('job_name', 'size'),
                                with_usage=('has_usage', 'sum'),
                                in_burst=('in_burst', 'sum'))
tab['usage_%'] = (100 * tab['with_usage'] / tab['jobs']).round(1)
tab['share_%'] = (100 * tab['jobs'] / tab['jobs'].sum()).round(1)
print('\n' + tab.reindex(['warmup', 'train', 'val', 'test']).to_string())
del use; gc.collect();

ea3832f (HEAD -> main, origin/main, origin/HEAD) fix: start train period on day 19 (sensor coverage)
PERIODS: {'warmup': (0.0, 19.0), 'train': (19.0, 45.0), 'val': (45.0, 52.0), 'test': (52.0, inf)}

period  group   jobs  run_time_median  run_time_p90  run_time_p99  in_burst
warmup   kept 123831            602.0       13143.0       76441.0         0
 train   kept 281064            476.0       12253.0       62007.0         0
 train purged    531          42194.0      204853.0      530297.0         0
   val   kept  88611            601.0       11222.0       55184.0         0
   val purged    801          35802.0      146534.0      366509.0         0
  test   kept 219435            526.0       10646.0       61845.0     19543

          jobs  with_usage  in_burst  usage_%  share_%
period                                                
warmup  123831        1509         0      1.2     17.4
train   281064      270830         0     96.4     39.4
val      88611       87492         0     98.7  

## 6. M1 features
Groups A–E (see FEATURE_GROUPS). History uses an as-of join: only events known strictly before submission. Leakage is covered by tests/test_features.py.

In [22]:
# Pull, check the group_tag schema, run the leakage tests
!git -C {REPO} pull -q
!git -C {REPO} log --oneline -1
from tevzi.data.loader import table_columns
gt_cols = table_columns('group_tag', INTERIM)
print('group_tag ->', gt_cols)
assert {'inst_id', 'group', 'workload'} <= set(gt_cols), 'group_tag columns differ: send this output'
!cd {REPO} && PYTHONPATH=src python -m pytest -q tests/test_features.py

a706c5c (HEAD -> main, origin/main, origin/HEAD) fix: join group_tag via inst_id
group_tag -> ['inst_id', 'user', 'gpu_type_spec', 'group', 'workload']
.......                                                                  [100%]
7 passed in 0.78s


In [23]:
# Build M1 features for the model set; check history coverage per period and a first signal check
from tevzi.features.m1 import build_features, FEATURE_GROUPS
t0 = time.time()
feats = build_features(jobs, targets, INTERIM)
gc.collect()
print(f'done in {time.time() - t0:.0f} s | shape: {feats.shape} | '
      f'RAM used: {psutil.virtual_memory().used / 1e9:.2f} GB')
assert len(feats) == len(targets) and feats['job_name'].is_unique

f = feats.merge(targets[['job_name', 'period', 'purged', 'log_run_time']], on='job_name')
f = f[~f['purged']]
cov = f.groupby('period').agg(
    grp_seen=('grp_seen_before', 'mean'),
    grp_rt_hist=('grp_rt_n', lambda s: (s > 0).mean()),
    usr_rt_hist=('usr_rt_n', lambda s: (s > 0).mean()),
    grp_use_hist=('grp_use_n', lambda s: (s > 0).mean()),
)
print('\nShare of jobs with history, per period:')
print((100 * cov).round(1).reindex(['warmup', 'train', 'val', 'test']).to_string())

tr = f[f['period'].eq('train')]
print('\nSpearman rho with log_run_time (train, jobs with group history):')
h = tr[tr['grp_rt_n'] > 0]
for c in ['grp_rt_last', 'grp_rt_last2', 'grp_rt_mean', 'usr_rt_last2']:
    print(f'  {c:14s} {h[c].corr(h["log_run_time"], method="spearman"):.3f}')
del f, tr, h; gc.collect();

done in 24 s | shape: (714273, 47) | RAM used: 3.72 GB

Share of jobs with history, per period:
        grp_seen  grp_rt_hist  usr_rt_hist  grp_use_hist
period                                                  
warmup      65.6         63.7         98.7           0.0
train       71.7         70.5         99.9          68.0
val         72.9         70.7         99.9          70.6
test        72.7         71.5         99.9          71.4

Spearman rho with log_run_time (train, jobs with group history):
  grp_rt_last    0.928
  grp_rt_last2   0.932
  grp_rt_mean    0.926
  usr_rt_last2   0.486


In [24]:
# Checkpoint: save the job flags, targets and M1 features to Drive
PROC = DRIVE / 'data/processed'
keep = ['job_name', 'user', 'status', 'submit_time', 'end_time', 'first_start', 'n_tasks',
        'req_gpu_total', 'req_cpu_total', 'req_mem_total', 'R2_cpu_only',
        'm2_arrival', 'm2_occupancy', 'model', 'first_rule']
jobs[keep].to_parquet(PROC / 'jobs_flags.parquet', index=False)
targets.to_parquet(PROC / 'targets.parquet', index=False)
feats.to_parquet(PROC / 'm1_features.parquet', index=False)
for p in sorted(PROC.glob('*.parquet')):
    print(f'{p.name:22s} {os.path.getsize(p) / 1e6:6.1f} MB')

jobs_flags.parquet       40.7 MB
m1_features.parquet      77.7 MB
targets.parquet          36.1 MB


## 7. M2 cluster-state time series
Step = 5 min (decision 16); last step = exact state at submission. Exploration first: how much memory does the cluster state carry (choice of window N)?

In [25]:
# Exploration: requested-GPU occupancy and arrivals on a 5-min grid; autocorrelation
import numpy as np
from tevzi.data.loader import load_table
from tevzi.data.split import T0
STEP = 300
gpu_cap = 100 * load_table('machine_spec', INTERIM, columns=['cap_gpu'])['cap_gpu'].sum()
n_bins = int((jobs['end_time'].max() - T0) // STEP) + 2

def to_bin(t):
    return np.clip(((t - T0) // STEP).astype('int64'), 0, n_bins - 1).to_numpy()

occ = jobs[jobs['m2_occupancy']]
delta = np.zeros(n_bins + 1)
np.add.at(delta, to_bin(occ['first_start']), occ['req_gpu_total'].to_numpy(dtype=float))
np.add.at(delta, to_bin(occ['end_time']), -occ['req_gpu_total'].to_numpy(dtype=float))
gpu_occ = pd.Series(np.cumsum(delta)[:n_bins] / gpu_cap)
arr = jobs[jobs['m2_arrival']]
arrivals = pd.Series(np.bincount(to_bin(arr['submit_time']), minlength=n_bins)[:n_bins].astype(float))
del occ, arr, delta; gc.collect()

print(f'grid: {n_bins:,} bins | GPU capacity: {gpu_cap / 100:,.0f} GPUs | '
      f'requested-GPU occupancy: median {gpu_occ.median():.2f}, p99 {gpu_occ.quantile(0.99):.2f}')
lags = {'5m': 1, '30m': 6, '1h': 12, '2h': 24, '6h': 72, '12h': 144, '24h': 288}
print('\nautocorrelation')
print('lag      ' + ''.join(f'{k:>7s}' for k in lags))
for name, s in [('gpu_occ', gpu_occ), ('arrivals', arrivals)]:
    print(f'{name:9s}' + ''.join(f'{s.autocorr(l):7.2f}' for l in lags.values()))

grid: 19,858 bins | GPU capacity: 6,742 GPUs | requested-GPU occupancy: median 0.44, p99 0.63

autocorrelation
lag           5m    30m     1h     2h     6h    12h    24h
gpu_occ     1.00   0.99   0.97   0.94   0.83   0.73   0.84
arrivals    0.71   0.60   0.53   0.46   0.25   0.07   0.37


In [26]:
# Exploration: Spearman rho between log_run_time and the mean state over the w before submission
# (train jobs; only grid steps strictly before the submission step are used)
tr = targets[targets['period'].eq('train') & ~targets['purged']][['job_name', 'log_run_time']]
tr = tr.merge(jobs[['job_name', 'submit_time']], on='job_name').reset_index(drop=True)
b = to_bin(tr['submit_time'])
cs = np.concatenate([[0.0], np.cumsum(gpu_occ.to_numpy())])
ca = np.concatenate([[0.0], np.cumsum(arrivals.to_numpy())])
print('window   rho(gpu_occ)   rho(arrivals)')
for k, w in {'5m': 1, '1h': 12, '2h': 24, '6h': 72, '12h': 144, '24h': 288}.items():
    lo = np.clip(b - w, 0, None)
    n_ = np.maximum(b - lo, 1)
    occ_w = pd.Series((cs[b] - cs[lo]) / n_)
    arr_w = pd.Series((ca[b] - ca[lo]) / n_)
    print(f'{k:>6s}   {occ_w.corr(tr["log_run_time"], method="spearman"):+12.3f}   '
          f'{arr_w.corr(tr["log_run_time"], method="spearman"):+13.3f}')
del tr, b, cs, ca; gc.collect();

window   rho(gpu_occ)   rho(arrivals)
    5m         -0.018          -0.040
    1h         -0.016          -0.026
    2h         -0.016          -0.023
    6h         -0.015          -0.027
   12h         -0.006          -0.025
   24h         -0.003          -0.044


In [27]:
# Does the cluster state explain what job history does not? (train jobs)
tr = targets[targets['period'].eq('train') & ~targets['purged']][['job_name', 'log_run_time']]
tr = (tr.merge(jobs[['job_name', 'submit_time']], on='job_name')
        .merge(feats[['job_name', 'grp_rt_n', 'grp_rt_last2', 'usr_rt_last2']], on='job_name')
        .reset_index(drop=True))
b = to_bin(tr['submit_time'])
cs = np.concatenate([[0.0], np.cumsum(gpu_occ.to_numpy())])
ca = np.concatenate([[0.0], np.cumsum(arrivals.to_numpy())])

has = tr['grp_rt_n'] > 0
res_g = tr['log_run_time'] - tr['grp_rt_last2']      # residual vs group history
res_u = tr['log_run_time'] - tr['usr_rt_last2']      # residual vs user history (jobs without group history)
print(f'jobs with group history: {has.sum():,} | without: {(~has).sum():,}\n')
print('         with group history           without group history')
print('window   rho(res,occ)  rho(res,arr)   rho(res_u,occ)  rho(res_u,arr)')
for k, w in {'5m': 1, '1h': 12, '2h': 24, '6h': 72, '12h': 144, '24h': 288}.items():
    lo = np.clip(b - w, 0, None)
    n_ = np.maximum(b - lo, 1)
    occ_w = pd.Series((cs[b] - cs[lo]) / n_)
    arr_w = pd.Series((ca[b] - ca[lo]) / n_)
    r = [occ_w[has].corr(res_g[has], method='spearman'), arr_w[has].corr(res_g[has], method='spearman'),
         occ_w[~has].corr(res_u[~has], method='spearman'), arr_w[~has].corr(res_u[~has], method='spearman')]
    print(f'{k:>6s}   {r[0]:+12.3f}  {r[1]:+12.3f}   {r[2]:+14.3f}  {r[3]:+14.3f}')
del tr, b, cs, ca, res_g, res_u; gc.collect();

jobs with group history: 198,050 | without: 83,014

         with group history           without group history
window   rho(res,occ)  rho(res,arr)   rho(res_u,occ)  rho(res_u,arr)
    5m         +0.027        +0.008           +0.010          +0.016
    1h         +0.024        +0.030           +0.017          +0.001
    2h         +0.022        +0.023           +0.021          -0.003
    6h         +0.014        +0.001           +0.028          +0.003
   12h         -0.005        -0.014           +0.012          -0.008
   24h         +0.004        +0.001           +0.013          -0.029


In [28]:
# Which resource carries the signal (GPU / CPU / memory), and is the effect non-linear?
spec = load_table('machine_spec', INTERIM, columns=['cap_cpu', 'cap_mem'])
cpu_cap, mem_cap = 100 * spec['cap_cpu'].sum(), spec['cap_mem'].sum()
occ = jobs[jobs['m2_occupancy']]
s_bin, e_bin = to_bin(occ['first_start']), to_bin(occ['end_time'])

def occupancy(col, cap):
    d = np.zeros(n_bins + 1)
    v = occ[col].to_numpy(dtype=float)
    np.add.at(d, s_bin, v)
    np.add.at(d, e_bin, -v)
    return np.cumsum(d)[:n_bins] / cap

series = {'gpu': gpu_occ.to_numpy(),
          'cpu': occupancy('req_cpu_total', cpu_cap),
          'mem': occupancy('req_mem_total', mem_cap)}
del occ, s_bin, e_bin; gc.collect()

tr = targets[targets['period'].eq('train') & ~targets['purged']][['job_name', 'log_run_time']]
tr = (tr.merge(jobs[['job_name', 'submit_time']], on='job_name')
        .merge(feats[['job_name', 'grp_rt_n', 'grp_rt_last2', 'usr_rt_last2']], on='job_name')
        .reset_index(drop=True))
prev = np.clip(to_bin(tr['submit_time']) - 1, 0, None)        # last full grid step before submission
has = tr['grp_rt_n'] > 0
res_g = tr['log_run_time'] - tr['grp_rt_last2']
res_u = tr['log_run_time'] - tr['usr_rt_last2']

print('resource  median_occ  p99_occ   rho(res,occ) with hist   rho(res_u,occ) without')
for name, s in series.items():
    last = pd.Series(s[prev])
    print(f'{name:8s}  {np.median(s):10.2f}  {np.quantile(s, 0.99):7.2f}   '
          f'{last[has].corr(res_g[has], method="spearman"):+22.3f}   '
          f'{last[~has].corr(res_u[~has], method="spearman"):+20.3f}')

print('\nMean residual (with group history) by occupancy decile, low -> high:')
for name in ['gpu', 'cpu', 'mem']:
    last = pd.Series(series[name][prev])[has]
    dec = pd.qcut(last, 10, labels=False, duplicates='drop')
    print(f'{name}: ' + ' '.join(f'{v:+.2f}' for v in res_g[has].groupby(dec).mean()))
del tr, prev, res_g, res_u; gc.collect();

resource  median_occ  p99_occ   rho(res,occ) with hist   rho(res_u,occ) without
gpu             0.44     0.63                   +0.027                 +0.010
cpu             0.26     0.39                   +0.030                 +0.007
mem             0.17     0.26                   +0.024                 -0.001

Mean residual (with group history) by occupancy decile, low -> high:
gpu: -0.01 -0.06 -0.02 -0.01 +0.03 -0.06 -0.07 -0.03 -0.04 +0.00
cpu: +0.02 -0.07 -0.03 +0.00 -0.05 -0.07 -0.04 -0.04 +0.02 -0.02
mem: +0.02 -0.07 -0.01 -0.02 -0.03 -0.06 -0.07 -0.05 -0.01 +0.03


In [29]:
# Sanity check: does the cluster state explain waiting? And the usage targets' residuals?
tr = targets[targets['period'].eq('train') & ~targets['purged']][
    ['job_name', 'wait_time', 'gpu_ratio', 'mem_ratio', 'has_usage']]
tr = (tr.merge(jobs[['job_name', 'submit_time']], on='job_name')
        .merge(feats[['job_name', 'grp_use_n', 'grp_gpu_mean', 'grp_mem_mean']], on='job_name')
        .reset_index(drop=True))
prev = np.clip(to_bin(tr['submit_time']) - 1, 0, None)
u = tr['has_usage'] & (tr['grp_use_n'] > 0)
res_gpu = tr['gpu_ratio'] - tr['grp_gpu_mean']
res_mem = tr['mem_ratio'] - tr['grp_mem_mean']
arr_last = pd.Series(arrivals.to_numpy()[prev])

print(f'jobs: {len(tr):,} | with usage + group usage history: {u.sum():,}\n')
print('signal     rho(wait,·)   rho(gpu_res,·)   rho(mem_res,·)')
for name, s in [('gpu_occ', series['gpu']), ('cpu_occ', series['cpu']), ('mem_occ', series['mem'])]:
    last = pd.Series(s[prev])
    print(f'{name:8s}  {last.corr(tr["wait_time"], method="spearman"):+11.3f}   '
          f'{last[u].corr(res_gpu[u], method="spearman"):+14.3f}   '
          f'{last[u].corr(res_mem[u], method="spearman"):+14.3f}')
print(f'{"arrivals":8s}  {arr_last.corr(tr["wait_time"], method="spearman"):+11.3f}   '
      f'{arr_last[u].corr(res_gpu[u], method="spearman"):+14.3f}   '
      f'{arr_last[u].corr(res_mem[u], method="spearman"):+14.3f}')

print('\nMedian wait (s) by GPU occupancy decile, low -> high:')
dec = pd.qcut(pd.Series(series['gpu'][prev]), 10, labels=False, duplicates='drop')
print(' '.join(f'{v:.0f}' for v in tr['wait_time'].groupby(dec).median()))
del tr, prev, u, res_gpu, res_mem, arr_last; gc.collect();

jobs: 281,064 | with usage + group usage history: 184,962

signal     rho(wait,·)   rho(gpu_res,·)   rho(mem_res,·)
gpu_occ        -0.004           -0.010           +0.015
cpu_occ        -0.030           -0.018           +0.008
mem_occ        -0.035           -0.020           +0.010
arrivals       +0.026           -0.020           -0.009

Median wait (s) by GPU occupancy decile, low -> high:
8 7 7 7 7 7 6 6 6 7


In [30]:
# Pool-aware state: occupancy per GPU-type pool (actual placement) and queued demand per requested type
spec = load_table('machine_spec', INTERIM, columns=['machine', 'gpu_type', 'cap_gpu'])
pool_cap = (100 * spec.groupby('gpu_type')['cap_gpu'].sum()).drop('CPU', errors='ignore')
print('pool capacity (GPUs):', (pool_cap / 100).astype(int).to_dict())

occ = jobs.loc[jobs['m2_occupancy'] & ~jobs['R2_cpu_only'],
               ['job_name', 'first_start', 'end_time', 'req_gpu_total']]
inst = load_table('instance', INTERIM, columns=['job_name', 'machine'])
inst = inst[inst['job_name'].isin(occ['job_name'])].dropna(subset=['machine']).drop_duplicates('job_name')
inst = inst.merge(spec[['machine', 'gpu_type']], on='machine', how='left')
occ = occ.merge(inst[['job_name', 'gpu_type']].rename(columns={'gpu_type': 'pool'}), on='job_name', how='left')
del inst; gc.collect()
print('occupancy jobs by actual pool:', occ['pool'].value_counts(dropna=False).to_dict())

def series_from(start, end, weight):
    d = np.zeros(n_bins + 1)
    np.add.at(d, to_bin(start), weight)
    np.add.at(d, to_bin(end), -weight)
    return np.cumsum(d)[:n_bins]

pool_occ = {p: series_from(g['first_start'], g['end_time'], g['req_gpu_total'].to_numpy(dtype=float)) / pool_cap[p]
            for p, g in occ.groupby('pool') if p in pool_cap.index}
del occ; gc.collect()

q = targets[['job_name', 'main_gpu_type']].merge(
    jobs[['job_name', 'submit_time', 'first_start', 'req_gpu_total']], on='job_name')
queue = {p: series_from(g['submit_time'], g['first_start'], g['req_gpu_total'].to_numpy(dtype=float))
         for p, g in q.groupby('main_gpu_type')}
del q; gc.collect()
print('\npool       occ median   occ p99')
for p, s in pool_occ.items():
    print(f'{p:8s}  {np.median(s):10.2f}  {np.quantile(s, 0.99):8.2f}')
print(f'\nRAM used: {psutil.virtual_memory().used / 1e9:.2f} GB')

pool capacity (GPUs): {'MISC': 2240, 'P100': 1596, 'T4': 994, 'V100': 832, 'V100M32': 1080}
occupancy jobs by actual pool: {'MISC': 390953, 'T4': 181223, 'P100': 70504, 'CPU': 35601, 'V100': 24276, 'V100M32': 12256}

pool       occ median   occ p99
MISC            0.23      0.39
P100            0.47      0.96
T4              0.38      1.03
V100            0.28      0.68
V100M32         0.24      0.53

RAM used: 3.58 GB


In [31]:
# Does the job's own pool state explain waiting or run-time residuals? (train jobs)
tr = targets[targets['period'].eq('train') & ~targets['purged']][
    ['job_name', 'main_gpu_type', 'wait_time', 'log_run_time']]
tr = (tr.merge(jobs[['job_name', 'submit_time']], on='job_name')
        .merge(feats[['job_name', 'grp_rt_n', 'grp_rt_last2']], on='job_name')
        .reset_index(drop=True))
prev = np.clip(to_bin(tr['submit_time']) - 1, 0, None)
o = np.full(len(tr), np.nan)
pr = np.full(len(tr), np.nan)
for p, s in pool_occ.items():
    m = tr['main_gpu_type'].eq(p).to_numpy()
    o[m] = s[prev[m]]
    free = np.maximum((1 - s[prev[m]]) * pool_cap[p], 100.0)    # at least one free GPU
    pr[m] = queue.get(p, np.zeros(n_bins))[prev[m]] / free
tr['pool_occ'], tr['pressure'] = o, pr
tr['res'] = tr['log_run_time'] - tr['grp_rt_last2']
tr['has'] = tr['grp_rt_n'] > 0

def rho(a, b):
    return a.corr(b, method='spearman')

print('type        jobs  rho(wait,occ)  rho(wait,press)  rho(res,occ)  rho(res,press)  wait_p90')
for p, g in [('ALL', tr)] + list(tr.groupby('main_gpu_type')):
    h = g[g['has']]
    print(f"{p:8s} {len(g):8,d}  {rho(g['pool_occ'], g['wait_time']):+13.3f}  {rho(g['pressure'], g['wait_time']):+15.3f}"
          f"  {rho(h['pool_occ'], h['res']):+12.3f}  {rho(h['pressure'], h['res']):+14.3f}  {g['wait_time'].quantile(0.9):8.0f}")

print('\np90 wait (s) by pool-occupancy decile, low -> high:')
for p in ['V100', 'V100M32', 'P100', 'T4', 'MISC']:
    g = tr[tr['main_gpu_type'].eq(p)]
    if len(g) < 1000:
        continue
    dec = pd.qcut(g['pool_occ'], 10, labels=False, duplicates='drop')
    print(f'{p:8s} ' + ' '.join(f'{v:.0f}' for v in g['wait_time'].groupby(dec).quantile(0.9)))
del tr, prev, o, pr; gc.collect();

type        jobs  rho(wait,occ)  rho(wait,press)  rho(res,occ)  rho(res,press)  wait_p90
ALL       281,064         -0.190           +0.174        +0.015          +0.004       114
MISC      171,214         +0.033           +0.036        +0.026          +0.010       109
P100       19,754         +0.039           +0.139        -0.020          -0.008       927
T4         78,989         +0.015           +0.055        +0.012          +0.006         4
V100        7,285         +0.082           +0.287        +0.014          +0.031      5214
V100M32     3,822         +0.090           +0.384        +0.021          +0.103      7602

p90 wait (s) by pool-occupancy decile, low -> high:
V100     5304 11374 3978 29907 3600 2186 2197 5763 23814 8440
V100M32  7628 6598 5996 3623 1558 375 13331 4346 11998 14581
P100     2465 1432 917 572 372 984 241 634 838 514
T4       4 4 4 4 4 4 4 4 4 30
MISC     68 49 106 68 106 107 99 115 122 454


In [32]:
# Pull, free exploration objects, run all leakage tests, build M2 (grid + per-job snapshots)
!git -C {REPO} pull -q
!git -C {REPO} log --oneline -1
for name in ['gpu_occ', 'arrivals', 'series', 'pool_occ', 'pool_cap', 'queue', 'spec']:
    globals().pop(name, None)
gc.collect()
!cd {REPO} && PYTHONPATH=src python -m pytest -q tests/test_features.py tests/test_m2.py

from tevzi.features.m2 import build_m2, windows, fit_scaler, CHANNELS, POOL_KEYS
t0 = time.time()
grid, snap, caps = build_m2(jobs, targets, INTERIM)
gc.collect()
print(f'done in {time.time() - t0:.0f} s | grid {grid.shape} | snapshots {snap.shape} | '
      f'RAM used: {psutil.virtual_memory().used / 1e9:.2f} GB')
print('capacities (GPUs):', {k: v / 100 for k, v in caps.items() if k not in ('cpu', 'mem')})
print('own pool of model jobs:', snap['pool_key'].value_counts().to_dict())
assert len(snap) == len(targets) and snap['job_name'].is_unique

4456819 (HEAD -> main, origin/main, origin/HEAD) feat: add pool-aware M2 cluster-state series with leakage tests
.............                                                            [100%]
13 passed in 0.59s
done in 15 s | grid (19858, 28) | snapshots (714273, 15) | RAM used: 3.62 GB
capacities (GPUs): {'gpu': 6742.0, 'ALL': 6742.0, 'P100': 1596.0, 'T4': 994.0, 'V100': 832.0, 'V100M32': 1080.0}
own pool of model jobs: {'ALL': 456357, 'T4': 183406, 'P100': 49366, 'V100': 16460, 'V100M32': 8684}


In [33]:
# Sanity checks: channel ranges in the training period; does the snapshot reproduce the exploration?
from tevzi.data.split import T0, DAY
tr_rows = grid['t'].between(T0 + 19 * DAY, T0 + 45 * DAY)
print('Grid channels in train period (median / p99):')
for c in ['arr_count', 'arr_gpu', 'occ_gpu', 'occ_cpu', 'occ_mem', 'pressure', 'use_gpu']:
    print(f'  {c:10s} {grid.loc[tr_rows, c].median():8.3f} / {grid.loc[tr_rows, c].quantile(0.99):8.3f}')
for key in POOL_KEYS:
    pc, pp = grid.loc[tr_rows, f'{key}:pool_occ'], grid.loc[tr_rows, f'{key}:pool_pressure']
    print(f'  {key:8s} pool_occ {pc.median():.2f} / {pc.quantile(0.99):.2f} | '
          f'pool_pressure {pp.median():.3f} / {pp.quantile(0.99):.3f}')

s = snap.merge(targets[['job_name', 'period', 'purged', 'wait_time']], on='job_name')
s = s[s['period'].eq('train') & ~s['purged']]
print('\nSnapshot pool_pressure vs wait (train, Spearman):')
for key, g in s.groupby('pool_key'):
    print(f"  {key:8s} n={len(g):7,d}  rho={g['pool_pressure'].corr(g['wait_time'], method='spearman'):+.3f}")

X, valid = windows(grid, s, n_steps=24)
print(f'\ntrain windows: {X.shape} ({X.nbytes / 1e9:.2f} GB) | padded steps: {(~valid).mean():.4%}')
print('finite:', bool(np.isfinite(X).all()))
del s, X, valid; gc.collect();

Grid channels in train period (median / p99):
  arr_count     3.829 /    5.362
  arr_gpu       0.019 /    0.075
  occ_gpu       0.449 /    0.624
  occ_cpu       0.268 /    0.385
  occ_mem       0.174 /    0.255
  pressure      0.041 /    0.198
  use_gpu       0.171 /    0.253
  ALL      pool_occ 0.34 / 0.48 | pool_pressure 0.034 / 0.142
  P100     pool_occ 0.51 / 0.88 | pool_pressure 0.019 / 0.468
  T4       pool_occ 0.35 / 0.87 | pool_pressure 0.003 / 0.500
  V100     pool_occ 0.30 / 0.70 | pool_pressure 0.015 / 0.559
  V100M32  pool_occ 0.27 / 0.55 | pool_pressure 0.005 / 0.235

Snapshot pool_pressure vs wait (train, Spearman):
  ALL      n=171,239  rho=+0.062
  P100     n= 19,729  rho=+0.169
  T4       n= 78,989  rho=+0.089
  V100     n=  7,285  rho=+0.377
  V100M32  n=  3,822  rho=+0.384

train windows: (281064, 25, 11) (0.31 GB) | padded steps: 0.0000%
finite: True


In [34]:
# Pull the fix, reload M2, rebuild, and check that the ALL pool now matches cluster GPU occupancy
!git -C {REPO} pull -q
!git -C {REPO} log --oneline -1
import importlib
import tevzi.features.m2 as m2mod
importlib.reload(m2mod)
del grid, snap; gc.collect()
t0 = time.time()
grid, snap, caps = m2mod.build_m2(jobs, targets, INTERIM)
print(f'done in {time.time() - t0:.0f} s')

tr_rows = grid['t'].between(T0 + 19 * DAY, T0 + 45 * DAY)
print(f"train median: occ_gpu {grid.loc[tr_rows, 'occ_gpu'].median():.3f} | "
      f"ALL:pool_occ {grid.loc[tr_rows, 'ALL:pool_occ'].median():.3f}")
for key in m2mod.POOL_KEYS[1:]:
    pc = grid.loc[tr_rows, f'{key}:pool_occ']
    print(f'  {key:8s} pool_occ median {pc.median():.2f} / p99 {pc.quantile(0.99):.2f}')

s = snap.merge(targets[['job_name', 'period', 'purged', 'wait_time']], on='job_name')
s = s[s['period'].eq('train') & ~s['purged']]
print('\nSnapshot pool_pressure vs wait (train, Spearman):')
for key, g in s.groupby('pool_key'):
    print(f"  {key:8s} rho={g['pool_pressure'].corr(g['wait_time'], method='spearman'):+.3f}")
del s; gc.collect();

9a080f9 (HEAD -> main, origin/main, origin/HEAD) fix: assign M2 pool by the GPU machines a job ran on
done in 18 s
train median: occ_gpu 0.449 | ALL:pool_occ 0.449
  P100     pool_occ median 0.41 / p99 0.70
  T4       pool_occ median 0.36 / p99 0.81
  V100     pool_occ median 0.26 / p99 0.64
  V100M32  pool_occ median 0.27 / p99 0.62

Snapshot pool_pressure vs wait (train, Spearman):
  ALL      rho=+0.061
  P100     rho=+0.159
  T4       rho=+0.089
  V100     rho=+0.372
  V100M32  rho=+0.380


In [35]:
# Sensitivity (concession i): add Failed jobs' load to the endogenous channels and compare with D
t_grid = grid['t'].to_numpy()
tr_rows = grid['t'].between(T0 + 19 * DAY, T0 + 45 * DAY).to_numpy()
cap = caps['gpu']

fail = jobs[jobs['status'].eq('Failed') & jobs['first_start'].notna()].copy()
end_known = fail['end_time'].fillna(fail['last_end']) if 'last_end' in fail else fail['end_time']
fail['end_known'] = end_known.where(end_known > fail['first_start'])
med_run = (fail['end_known'] - fail['first_start']).median()
fail['end_imp'] = fail['end_known'].fillna(fail['first_start'] + med_run)
fail['gpu'] = fail['req_gpu_total'].fillna(0)
print(f"Failed jobs: {len(fail):,} | known end: {fail['end_known'].notna().mean():.1%} | "
      f"median known run: {med_run:.0f} s")

base = jobs[jobs['m2_occupancy']]
occ_D = grid['occ_gpu'].to_numpy()
queue_D = m2mod.Level(base['submit_time'], base['first_start'], base['req_gpu_total'].fillna(0))(t_grid)
queue_F = m2mod.Level(fail['submit_time'], fail['first_start'], fail['gpu'])(t_grid)

rows = []
for name, end in [('a_known_end', 'end_known'), ('b_imputed', 'end_imp')]:
    f = fail.dropna(subset=[end])
    occ_v = occ_D + m2mod.Level(f['first_start'], f[end], f['gpu'])(t_grid) / cap
    press_D = queue_D / np.maximum(cap - occ_D * cap, 100.0)
    press_v = (queue_D + queue_F) / np.maximum(cap - occ_v * cap, 100.0)
    for ch, d, v in [('occ_gpu', occ_D, occ_v), ('pressure', press_D, press_v)]:
        d_, v_ = d[tr_rows], v[tr_rows]
        rows.append({'variant': name, 'channel': ch,
                     'median_D': np.median(d_), 'median_variant': np.median(v_),
                     'pearson': np.corrcoef(d_, v_)[0, 1],
                     'spearman': pd.Series(d_).corr(pd.Series(v_), method='spearman')})
sens = pd.DataFrame(rows)
print(sens.round(3).to_string(index=False))
sens.to_csv(DRIVE / 'results/tables/m2_sensitivity_failed.csv', index=False)
del fail, base, queue_D, queue_F; gc.collect();

Failed jobs: 256,555 | known end: 13.8% | median known run: 183 s
    variant  channel  median_D  median_variant  pearson  spearman
a_known_end  occ_gpu     0.449           0.544    0.932     0.930
a_known_end pressure     0.041           0.123    0.790     0.862
  b_imputed  occ_gpu     0.449           0.545    0.933     0.931
  b_imputed pressure     0.041           0.123    0.789     0.861


In [37]:
# Independent check (decision 11): our usage-based GPU series vs machine_metric GPU utilisation
mm = load_table('machine_metric', INTERIM,
                columns=['start_time', 'end_time', 'machine_gpu', 'machine_num_worker'])
print(f"rows: {len(mm):,} | missing machine_gpu: {mm['machine_gpu'].isna().mean():.1%} | "
      f"missing num_worker: {mm['machine_num_worker'].isna().mean():.1%}")
mm = mm.dropna(subset=['machine_gpu', 'start_time', 'end_time'])
w = mm['machine_gpu'] / mm['machine_num_worker'].fillna(1).clip(lower=1)   # each worker's share
assert np.isfinite(w).all()
mm_series = m2mod.Level(mm['start_time'], mm['end_time'], w)(t_grid) / cap
del mm, w; gc.collect()

ours, ref = grid['use_gpu'].to_numpy()[tr_rows], mm_series[tr_rows]
assert np.isfinite(ref).all()
day = ((t_grid[tr_rows] - T0) // DAY).astype(int)
daily = pd.DataFrame({'ours': ours, 'ref': ref, 'day': day}).groupby('day').mean()
print(f'medians: ours {np.median(ours):.3f} | machine_metric {np.median(ref):.3f}')
print(f'5-min series: pearson {np.corrcoef(ours, ref)[0, 1]:.3f} | '
      f"spearman {pd.Series(ours).corr(pd.Series(ref), method='spearman'):.3f}")
print(f"daily means:  pearson {daily['ours'].corr(daily['ref']):.3f} (n={len(daily)} days)")

rows: 2,009,423 | missing machine_gpu: 12.0% | missing num_worker: 14.7%
medians: ours 0.171 | machine_metric 0.109
5-min series: pearson -0.051 | spearman 0.173
daily means:  pearson -0.473 (n=27 days)


In [38]:
# Diagnose: does machine_metric follow the Failed-inclusive load rather than our Terminated-only series?
failed_load = occ_v - occ_D                       # variant b minus D (requested GPU of Failed jobs)
cands = {
    'ours_use_gpu (Terminated, usage)': grid['use_gpu'].to_numpy(),
    'occ_D (Terminated, requested)': occ_D,
    'occ_with_failed (requested)': occ_v,
    'failed_only (requested)': failed_load,
}
d = pd.DataFrame({k: v[tr_rows] for k, v in cands.items()})
d['machine_metric'] = mm_series[tr_rows]
d['day'] = day
daily = d.groupby('day').mean()
print('correlation with machine_metric      5-min spearman   daily pearson')
for k in cands:
    print(f'  {k:34s} {d[k].corr(d["machine_metric"], method="spearman"):+14.3f}'
          f'   {daily[k].corr(daily["machine_metric"]):+13.3f}')
print('\nDaily means (first 10 train days):')
print(daily[['ours_use_gpu (Terminated, usage)', 'failed_only (requested)', 'machine_metric']]
      .head(10).round(3).to_string())
del d; gc.collect();

correlation with machine_metric      5-min spearman   daily pearson
  ours_use_gpu (Terminated, usage)           +0.173          -0.473
  occ_D (Terminated, requested)              +0.445          -0.078
  occ_with_failed (requested)                +0.583          +0.305
  failed_only (requested)                    +0.563          +0.633

Daily means (first 10 train days):
     ours_use_gpu (Terminated, usage)  failed_only (requested)  machine_metric
day                                                                           
19                              0.203                    0.041           0.000
20                              0.208                    0.073           0.001
21                              0.201                    0.058           0.008
22                              0.176                    0.052           0.009
23                              0.191                    0.062           0.008
24                              0.201                    0.108         

In [39]:
# Find where machine_metric coverage starts, then re-run the comparison only where it measures
d = pd.DataFrame({k: v[tr_rows] for k, v in cands.items()})
d['machine_metric'] = mm_series[tr_rows]
d['day'] = day
daily = d.groupby('day').mean()
print('machine_metric daily mean (train days):')
print(' '.join(f"d{k}:{v:.2f}" for k, v in daily['machine_metric'].items()))

ref_start = int(daily.index[daily['machine_metric'] > 0.5 * daily['machine_metric'].median()].min())
print(f'\nreference coverage starts on day {ref_start} (first day above half its median)')
dd, dl = d[d['day'] >= ref_start], daily[daily.index >= ref_start]
print(f'\ncorrelation with machine_metric, days >= {ref_start}   5-min spearman   daily pearson (n={len(dl)})')
for k in cands:
    print(f'  {k:34s} {dd[k].corr(dd["machine_metric"], method="spearman"):+14.3f}'
          f'   {dl[k].corr(dl["machine_metric"]):+13.3f}')
del d, dd; gc.collect();

machine_metric daily mean (train days):
d19:0.00 d20:0.00 d21:0.01 d22:0.01 d23:0.01 d24:0.09 d25:0.11 d26:0.11 d27:0.13 d28:0.12 d29:0.11 d30:0.11 d31:0.11 d32:0.11 d33:0.12 d34:0.11 d35:0.11 d36:0.12 d37:0.12 d38:0.10 d39:0.09 d40:0.11 d41:0.13 d42:0.14 d43:0.14 d44:0.15 d45:0.11

reference coverage starts on day 24 (first day above half its median)

correlation with machine_metric, days >= 24   5-min spearman   daily pearson (n=22)
  ours_use_gpu (Terminated, usage)           +0.626          +0.206
  occ_D (Terminated, requested)              +0.752          +0.468
  occ_with_failed (requested)                +0.765          +0.587
  failed_only (requested)                    +0.340          +0.441


In [40]:
# Save M2: grid, per-job snapshots and capacities
import json
grid.to_parquet(PROC / 'm2_grid.parquet', index=False)
snap.to_parquet(PROC / 'm2_snapshots.parquet', index=False)
with open(PROC / 'm2_capacities.json', 'w') as fh:
    json.dump(caps, fh, indent=2)
for p in sorted(PROC.iterdir()):
    print(f'{p.name:24s} {os.path.getsize(p) / 1e6:7.1f} MB')

jobs_flags.parquet          40.7 MB
m1_features.parquet         77.7 MB
m2_capacities.json           0.0 MB
m2_grid.parquet              2.6 MB
m2_snapshots.parquet        56.8 MB
targets.parquet             36.1 MB
